In [1]:
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
import os
print(os.getcwd())

c:\Users\mpalazzo\Documents\progetto_boolean\notebook_files


```sql
SELECT
  orders.order_id,
  distribution_centers.name AS distribution_center_name,
  DATE_DIFF(DATE(orders.delivered_at), DATE(orders.created_at), DAY) AS delivery_days
FROM `bigquery-public-data.thelook_ecommerce.orders` AS orders
JOIN `bigquery-public-data.thelook_ecommerce.order_items` AS order_items
  ON orders.order_id = order_items.order_id
JOIN `bigquery-public-data.thelook_ecommerce.inventory_items` AS inventory_items
  ON order_items.inventory_item_id = inventory_items.id
JOIN `bigquery-public-data.thelook_ecommerce.distribution_centers` AS distribution_centers
  ON inventory_items.product_distribution_center_id = distribution_centers.id
WHERE orders.status = 'Complete'
  AND orders.delivered_at IS NOT NULL
  AND orders.created_at IS NOT NULL;
```

In [3]:
df = pd.read_csv(r'C:\Users\mpalazzo\Documents\progetto_boolean\definitive_csv_data\Distribution_center_orders.csv')

In [4]:
df

,order_id,distribution_center_name,delivery_days
0,712,Memphis TN,0
1,733,Memphis TN,0
2,733,Savannah GA,0
3,2314,Mobile AL,0
4,2571,Savannah GA,0
...,...,...,...
45125,123626,Port Authority of New York/New Jersey NY/NJ,8
45126,123911,Charleston SC,8
45127,123911,Houston TX,8
45128,123945,Savannah GA,8


In [5]:
#statistiche per ogni categoria
df_dc_stat=df.groupby('distribution_center_name')['delivery_days'].agg(['mean', 'std', 'count'])
df_dc_stat

,mean,std,count
distribution_center_name,,,
Charleston SC,3.944861,1.720802,4135
Chicago IL,3.993428,1.713744,5934
Houston TX,3.978388,1.721038,5645
Los Angeles CA,4.025352,1.711751,4260
Memphis TN,4.004438,1.719949,6084
Mobile AL,4.000657,1.738550,4567
New Orleans LA,3.999103,1.720788,3344
Philadelphia PA,4.014332,1.735840,4047
Port Authority of New York/New Jersey NY/NJ,4.017996,1.728513,4112


In [6]:

df['distribution_center_name'].unique()

array(['Memphis TN', 'Savannah GA', 'Mobile AL', 'Houston TX',
       'Chicago IL', 'Philadelphia PA',
       'Port Authority of New York/New Jersey NY/NJ', 'New Orleans LA',
       'Los Angeles CA', 'Charleston SC'], dtype=object)

In [7]:
memphis      = df[df['distribution_center_name'] == 'Memphis TN']['delivery_days'].to_numpy()
savannah     = df[df['distribution_center_name'] == 'Savannah GA']['delivery_days'].to_numpy()
mobile       = df[df['distribution_center_name'] == 'Mobile AL']['delivery_days'].to_numpy()
houston      = df[df['distribution_center_name'] == 'Houston TX']['delivery_days'].to_numpy()
chicago      = df[df['distribution_center_name'] == 'Chicago IL']['delivery_days'].to_numpy()
philadelphia = df[df['distribution_center_name'] == 'Philadelphia PA']['delivery_days'].to_numpy()
port_authority = df[df['distribution_center_name'] == 'Port Authority of New York/New Jersey NY/NJ']['delivery_days'].to_numpy()
new_orleans  = df[df['distribution_center_name'] == 'New Orleans LA']['delivery_days'].to_numpy()
los_angeles  = df[df['distribution_center_name'] == 'Los Angeles CA']['delivery_days'].to_numpy()
charleston   = df[df['distribution_center_name'] == 'Charleston SC']['delivery_days'].to_numpy()

In [8]:
f = stats.f_oneway(memphis, savannah,mobile,houston, chicago, philadelphia, port_authority,new_orleans, los_angeles, charleston)

In [9]:
print(f'F-statistic: {f.statistic}')
print(f'P-value [%]: {f.pvalue*100:.10f}')

if f.pvalue < 0.05:
    print('p-value < 5%: rifiutiamo l\'ipotesi nulla.')
    print(' Almeno un centro di distribuzione ha un tempo medio di consegna significativamente diverso.')
else:
    print('p-value ≥ 5%: non rifiutiamo l\'ipotesi nulla.')
    print('Nessuna differenza significativa trovata tra i centri di distribuzione.')

F-statistic: 0.8392868791915445
P-value [%]: 57.9687066148
p-value ≥ 5%: non rifiutiamo l'ipotesi nulla.
Nessuna differenza significativa trovata tra i centri di distribuzione.


In [10]:
import numpy as np

mu = df['delivery_days'].mean()
print(f'Media globale (μ): {mu:.4f} giorni')
print()

# Media di ogni gruppo
memphis_mean      = memphis.mean()
savannah_mean     = savannah.mean()
mobile_mean       = mobile.mean()
houston_mean      = houston.mean()
chicago_mean      = chicago.mean()
philadelphia_mean = philadelphia.mean()
port_authority_mean    = port_authority.mean()
new_orleans_mean  = new_orleans.mean()
los_angeles_mean  = los_angeles.mean()
charleston_mean   = charleston.mean()

print(f'Media Memphis:       {memphis_mean:.4f}')
print(f'Media Savannah:      {savannah_mean:.4f}')
print(f'Media Mobile:        {mobile_mean:.4f}')
print(f'Media Houston:       {houston_mean:.4f}')
print(f'Media Chicago:       {chicago_mean:.4f}')
print(f'Media Philadelphia:  {philadelphia_mean:.4f}')
print(f'Media port authority: {port_authority_mean:.4f}')
print(f'Media New Orleans:   {new_orleans_mean:.4f}')
print(f'Media Los Angeles:   {los_angeles_mean:.4f}')
print(f'Media Charleston:    {charleston_mean:.4f}')
print()


SSbtg = 0
SSbtg += len(memphis)      * pow(memphis_mean      - mu, 2)
SSbtg += len(savannah)     * pow(savannah_mean     - mu, 2)
SSbtg += len(mobile)       * pow(mobile_mean       - mu, 2)
SSbtg += len(houston)      * pow(houston_mean      - mu, 2)
SSbtg += len(chicago)      * pow(chicago_mean      - mu, 2)
SSbtg += len(philadelphia) * pow(philadelphia_mean - mu, 2)
SSbtg += len(port_authority)    * pow(port_authority_mean    - mu, 2)
SSbtg += len(new_orleans)  * pow(new_orleans_mean  - mu, 2)
SSbtg += len(los_angeles)  * pow(los_angeles_mean  - mu, 2)
SSbtg += len(charleston)   * pow(charleston_mean   - mu, 2)

# Calculate within groups sum of squares
SSwtg = 0
SSwtg_memphis      = 0
SSwtg_savannah     = 0
SSwtg_mobile       = 0
SSwtg_houston      = 0
SSwtg_chicago      = 0
SSwtg_philadelphia = 0
SSwtg_port_authority = 0
SSwtg_new_orleans  = 0
SSwtg_los_angeles  = 0
SSwtg_charleston   = 0

for value in memphis:
    SSwtg_memphis      += pow(value - memphis_mean, 2)
for value in savannah:
    SSwtg_savannah     += pow(value - savannah_mean, 2)
for value in mobile:
    SSwtg_mobile       += pow(value - mobile_mean, 2)
for value in houston:
    SSwtg_houston      += pow(value - houston_mean, 2)
for value in chicago:
    SSwtg_chicago      += pow(value - chicago_mean, 2)
for value in philadelphia:
    SSwtg_philadelphia += pow(value - philadelphia_mean, 2)
for value in port_authority:
    SSwtg_port_authority += pow(value - port_authority_mean, 2)
for value in new_orleans:
    SSwtg_new_orleans  += pow(value - new_orleans_mean, 2)
for value in los_angeles:
    SSwtg_los_angeles  += pow(value - los_angeles_mean, 2)
for value in charleston:
    SSwtg_charleston   += pow(value - charleston_mean, 2)

SSwtg = (SSwtg_memphis + SSwtg_savannah + SSwtg_mobile + SSwtg_houston +
         SSwtg_chicago + SSwtg_philadelphia + SSwtg_port_authority +
         SSwtg_new_orleans + SSwtg_los_angeles + SSwtg_charleston)

print(f'SSbtg: {SSbtg:.4f}')
print(f'SSwtg: {SSwtg:.4f}')
print()

N_gruppi = 10
N_totale = len(df)
btg_dof  = N_gruppi - 1        
wtg_dof  = N_totale - N_gruppi 

MSbtg = SSbtg / btg_dof
MSwtg = SSwtg / wtg_dof


F_stat = MSbtg / MSwtg


Media globale (μ): 3.9952 giorni

Media Memphis:       4.0044
Media Savannah:      3.9684
Media Mobile:        4.0007
Media Houston:       3.9784
Media Chicago:       3.9934
Media Philadelphia:  4.0143
Media port authority: 4.0180
Media New Orleans:   3.9991
Media Los Angeles:   4.0254
Media Charleston:    3.9449

SSbtg: 22.4531
SSwtg: 134119.5035

